# Phase 13 — Ensemble DistilBERT + DistilRoBERTa

**Μοντέλα:**
- DistilBERT train+valid 20 epochs → Kaggle: 0.76064
- DistilRoBERTa train+valid 20 epochs → Kaggle: 0.73867

**Μέθοδος:** Weighted average — δοκιμάζουμε διαφορετικά βάρη

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

# DistilBERT probs (20 epochs, train+valid)
bert_hazard_probs  = np.load('npy/bert_fulldata_test_hazard_probs.npy')
bert_product_probs = np.load('npy/bert_fulldata_test_product_probs.npy')

# DistilRoBERTa probs (20 epochs, train+valid)
roberta_hazard_probs  = np.load('npy/roberta_test_hazard_probs.npy')
roberta_product_probs = np.load('npy/roberta_test_product_probs.npy')

print(f'BERT hazard probs:    {bert_hazard_probs.shape}')
print(f'RoBERTa hazard probs: {roberta_hazard_probs.shape}')

BERT hazard probs:    (997, 10)
RoBERTa hazard probs: (997, 10)


In [2]:
# Δοκιμή διαφορετικών βαρών
# Δεν έχουμε validation set για tuning, οπότε δοκιμάζουμε
# λογικά βάρη βάσει των Kaggle scores
# DistilBERT: 0.76064, RoBERTa: 0.73867
# Αναλογία: ~0.51 / 0.49 → δοκιμάζουμε γύρω από αυτό

weight_combinations = [
    (1.0, 0.0),   # BERT μόνο (baseline)
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),   # Equal weights
    (0.4, 0.6),
    (0.0, 1.0),   # RoBERTa μόνο (baseline)
]


for w_bert, w_roberta in weight_combinations:
    hazard_avg  = w_bert * bert_hazard_probs  + w_roberta * roberta_hazard_probs
    product_avg = w_bert * bert_product_probs + w_roberta * roberta_product_probs

    test_hazard  = le_hazard.inverse_transform(hazard_avg.argmax(axis=1))
    test_product = le_product.inverse_transform(product_avg.argmax(axis=1))

    filename = f'submission_bert{int(w_bert*10)}_roberta{int(w_roberta*10)}.csv'
    pd.DataFrame({
        'id': test['id'],
        'hazard-category': test_hazard,
        'product-category': test_product
    }).to_csv(filename, index=False)
    print(f'BERT={w_bert:.1f}, RoBERTa={w_roberta:.1f} → {filename}')

print('1. submission_bert7_roberta3.csv  (BERT=0.7, RoBERTa=0.3)')
print('2. submission_bert5_roberta5.csv  (Equal weights)')
print('3. submission_bert6_roberta4.csv  (BERT=0.6, RoBERTa=0.4)')

BERT=1.0, RoBERTa=0.0 → submission_bert10_roberta0.csv
BERT=0.8, RoBERTa=0.2 → submission_bert8_roberta2.csv
BERT=0.7, RoBERTa=0.3 → submission_bert7_roberta3.csv
BERT=0.6, RoBERTa=0.4 → submission_bert6_roberta4.csv
BERT=0.5, RoBERTa=0.5 → submission_bert5_roberta5.csv
BERT=0.4, RoBERTa=0.6 → submission_bert4_roberta6.csv
BERT=0.0, RoBERTa=1.0 → submission_bert0_roberta10.csv
1. submission_bert7_roberta3.csv  (BERT=0.7, RoBERTa=0.3)
2. submission_bert5_roberta5.csv  (Equal weights)
3. submission_bert6_roberta4.csv  (BERT=0.6, RoBERTa=0.4)
